# Scratch Studio Comments Explorer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/22552/kasotest/blob/main/ScratchCommentsExplorer.ipynb)

`22552/kasotest` の `第二プロジェクト.sqlite.zst` を取得し、Colab上で展開して **Gradio検索・分析UI** を起動します。

- FTS5全文検索 / LIKE部分一致
- ユーザー・期間・親コメント/返信フィルタ
- CSV出力
- 投稿数ランキング / 日別推移
- 読み取り専用SQL Playground
- `share=True` で一時公開URLを発行

> DBは読み取り専用で開きます。Colabのセッションが終了するとGradioの共有URLも停止します。


In [ ]:
!pip -q install "gradio>=6.20,<7" zstandard plotly

In [ ]:
from pathlib import Path
import requests
import zstandard as zstd

RAW_DB_URL = "https://raw.githubusercontent.com/22552/kasotest/main/%E7%AC%AC%E4%BA%8C%E3%83%97%E3%83%AD%E3%82%B8%E3%82%A7%E3%82%AF%E3%83%88.sqlite.zst"
ZST_PATH = Path("/content/comments.sqlite.zst")
DB_PATH = Path("/content/comments.sqlite")

def download_file(url: str, dst: Path):
    if dst.exists() and dst.stat().st_size > 0:
        print(f"[download] reuse {dst} ({dst.stat().st_size / 1024 / 1024:.1f} MiB)")
        return

    tmp = Path(str(dst) + ".tmp")
    tmp.unlink(missing_ok=True)
    print("[download] downloading SQLite.zst from GitHub...")

    try:
        with requests.get(url, stream=True, timeout=(15, 180)) as r:
            r.raise_for_status()
            total = int(r.headers.get("content-length") or 0)
            done = 0
            with tmp.open("wb") as f:
                for chunk in r.iter_content(chunk_size=4 * 1024 * 1024):
                    if not chunk:
                        continue
                    f.write(chunk)
                    done += len(chunk)
                    if total:
                        print(
                            f"\r  {done/1024/1024:.1f}/{total/1024/1024:.1f} MiB ({done/total:.0%})",
                            end="",
                        )
        print()
        tmp.replace(dst)
    except Exception:
        tmp.unlink(missing_ok=True)
        raise

def decompress(src: Path, dst: Path):
    # --long=31 で圧縮したZstdフレームは最大2 GiBのwindowを要求し得る。
    # Python zstandard の max_window_size は KiB 単位。
    tmp = Path(str(dst) + ".tmp")

    # 前回の失敗で残った不完全なSQLiteを絶対に再利用しない。
    dst.unlink(missing_ok=True)
    tmp.unlink(missing_ok=True)

    print("[zstd] decompressing...")
    dctx = zstd.ZstdDecompressor(max_window_size=2 * 1024 * 1024)

    try:
        with src.open("rb") as fin, tmp.open("wb") as fout:
            dctx.copy_stream(fin, fout)
        tmp.replace(dst)
    except Exception:
        tmp.unlink(missing_ok=True)
        dst.unlink(missing_ok=True)
        raise

    print(f"[zstd] done: {dst.stat().st_size / 1024 / 1024:.1f} MiB")

download_file(RAW_DB_URL, ZST_PATH)
decompress(ZST_PATH, DB_PATH)
print("[ready]", DB_PATH)


In [ ]:
import sqlite3
import tempfile
import pandas as pd
import plotly.express as px
import gradio as gr

def db_connect():
    con = sqlite3.connect(
        f"file:{DB_PATH}?mode=ro",
        uri=True,
        timeout=5,
        check_same_thread=False,
    )
    con.execute("PRAGMA query_only=ON")
    con.execute("PRAGMA busy_timeout=5000")
    return con

def normalize_fts_query(q: str) -> str:
    terms = [t for t in (q or "").strip().split() if t]
    return " AND ".join('\"' + t.replace('\"', '""') + '\"' for t in terms)

def search_comments(query, search_mode, user, start_date, end_date, kind, limit):
    limit = max(1, min(int(limit), 500))
    clauses, params = [], []

    if kind == "親コメント":
        clauses.append("c.is_reply = 0")
    elif kind == "返信":
        clauses.append("c.is_reply = 1")

    if (user or "").strip():
        clauses.append("c.user = ? COLLATE NOCASE")
        params.append(user.strip())

    if (start_date or "").strip():
        clauses.append("c.datetime >= ?")
        params.append(start_date.strip())

    if (end_date or "").strip():
        e = end_date.strip()
        if len(e) == 10:
            e += "T23:59:59.999999Z"
        clauses.append("c.datetime <= ?")
        params.append(e)

    q = (query or "").strip()

    try:
        with db_connect() as con:
            if q and search_mode in ("FTS5（簡単）", "FTS5構文"):
                fts_q = normalize_fts_query(q) if search_mode == "FTS5（簡単）" else q
                all_clauses = ["comments_fts MATCH ?"] + clauses
                sql = f"""
                    SELECT
                        c.id, c.parent_id, c.is_reply,
                        c.user, c.user_id, c.datetime, c.content,
                        bm25(comments_fts) AS score
                    FROM comments_fts
                    JOIN comments AS c ON c.id = comments_fts.rowid
                    WHERE {' AND '.join(all_clauses)}
                    ORDER BY score, c.datetime DESC
                    LIMIT ?
                """
                df = pd.read_sql_query(sql, con, params=[fts_q] + params + [limit])
            else:
                if q:
                    clauses.insert(0, "c.content LIKE ?")
                    params.insert(0, f"%{q}%")
                where = "WHERE " + " AND ".join(clauses) if clauses else ""
                sql = f"""
                    SELECT
                        c.id, c.parent_id, c.is_reply,
                        c.user, c.user_id, c.datetime, c.content
                    FROM comments AS c
                    {where}
                    ORDER BY c.datetime DESC
                    LIMIT ?
                """
                df = pd.read_sql_query(sql, con, params=params + [limit])
    except Exception as e:
        return pd.DataFrame(), f"❌ `{type(e).__name__}: {e}`", None

    if not df.empty:
        df["種別"] = df["is_reply"].map({0: "親", 1: "返信"})
        cols = ["id", "parent_id", "種別", "user", "user_id", "datetime", "content"]
        if "score" in df.columns:
            cols.append("score")
        df = df[cols]

    tmp = tempfile.NamedTemporaryFile(prefix="comments-", suffix=".csv", delete=False)
    tmp.close()
    df.to_csv(tmp.name, index=False, encoding="utf-8-sig")
    return df, f"✅ **{len(df):,} 件表示**（最大 {limit:,} 件）", tmp.name

def summary_stats():
    with db_connect() as con:
        row = con.execute("""
            SELECT
                COUNT(*) AS total,
                SUM(is_reply = 0) AS top_level,
                SUM(is_reply = 1) AS replies,
                COUNT(DISTINCT user) AS users,
                MIN(datetime) AS oldest,
                MAX(datetime) AS newest
            FROM comments
        """).fetchone()
    total, top_level, replies, users, oldest, newest = row
    return f"""
### DB概要
- 全行数: **{total:,}**
- 親コメント: **{top_level:,}**
- 返信: **{replies:,}**
- ユーザー数: **{users:,}**
- 期間: `{oldest}` ～ `{newest}`
"""

def top_users(n):
    n = max(10, min(int(n), 200))
    with db_connect() as con:
        return pd.read_sql_query("""
            SELECT user, COUNT(*) AS comments
            FROM comments
            GROUP BY user
            ORDER BY comments DESC
            LIMIT ?
        """, con, params=[n])

def daily_plot():
    with db_connect() as con:
        df = pd.read_sql_query("""
            SELECT substr(datetime, 1, 10) AS day, COUNT(*) AS comments
            FROM comments
            GROUP BY day
            ORDER BY day
        """, con)
    return px.line(df, x="day", y="comments", title="日別コメント数") if not df.empty else None

SQL_PRESETS = {
    "最近100件": """SELECT id, parent_id, user, datetime, content
FROM comments
ORDER BY datetime DESC
LIMIT 100;""",
    "投稿数 上位100ユーザー": """SELECT user, COUNT(*) AS comments
FROM comments
GROUP BY user
ORDER BY comments DESC
LIMIT 100;""",
    "返信が多い親コメント": """SELECT p.id, p.user, p.datetime, p.content, COUNT(r.id) AS replies
FROM comments AS p
JOIN comments AS r ON r.parent_id = p.id
WHERE p.is_reply = 0
GROUP BY p.id
ORDER BY replies DESC
LIMIT 100;""",
    "FTS5検索例": """SELECT c.id, c.user, c.datetime, c.content
FROM comments_fts
JOIN comments AS c ON c.id = comments_fts.rowid
WHERE comments_fts MATCH 'Scratch'
ORDER BY bm25(comments_fts)
LIMIT 100;""",
}

def load_preset(name):
    return SQL_PRESETS.get(name, "")

def run_readonly_sql(sql):
    text = (sql or "").strip()
    if not text:
        return pd.DataFrame(), "SQLを入力してください。"

    stripped = text.rstrip().rstrip(";").strip()
    if ";" in stripped:
        return pd.DataFrame(), "❌ 複数SQL文は実行できません。"

    first = stripped.split(None, 1)[0].upper() if stripped else ""
    if first not in {"SELECT", "WITH", "EXPLAIN"}:
        return pd.DataFrame(), "❌ 読み取り専用です。SELECT / WITH / EXPLAIN のみ実行できます。"

    con = None
    try:
        con = db_connect()
        calls = 0
        def progress():
            nonlocal calls
            calls += 1
            return 1 if calls > 100_000 else 0
        con.set_progress_handler(progress, 1000)

        cur = con.execute(stripped)
        columns = [d[0] for d in (cur.description or [])]
        rows = cur.fetchmany(1001)
        clipped = len(rows) > 1000
        rows = rows[:1000]
        df = pd.DataFrame(rows, columns=columns)
        note = f"✅ {len(df):,} 行表示" + ("（先頭1,000行まで）" if clipped else "")
        return df, note
    except Exception as e:
        return pd.DataFrame(), f"❌ `{type(e).__name__}: {e}`"
    finally:
        if con is not None:
            con.close()

with gr.Blocks(title="Scratch Studio Comments Explorer") as demo:
    gr.Markdown("""
# 🔎 Scratch Studio Comments Explorer
約100万件級のコメントDBを SQLite + FTS5 で検索・分析します。**DBは読み取り専用**です。
""")

    with gr.Tab("検索"):
        with gr.Row():
            query = gr.Textbox(label="本文検索", placeholder="例: Scratch")
            search_mode = gr.Dropdown(
                ["FTS5（簡単）", "FTS5構文", "部分一致（LIKE）"],
                value="FTS5（簡単）",
                label="検索方式",
            )
        with gr.Row():
            user = gr.Textbox(label="ユーザー（完全一致・大文字小文字無視）")
            kind = gr.Dropdown(["すべて", "親コメント", "返信"], value="すべて", label="種別")
        with gr.Row():
            start_date = gr.Textbox(label="開始日時", placeholder="2023-09-05")
            end_date = gr.Textbox(label="終了日時", placeholder="2026-08-20")
            limit = gr.Slider(10, 500, value=100, step=10, label="最大表示件数")
        search_btn = gr.Button("検索", variant="primary")
        status = gr.Markdown()
        results = gr.Dataframe(label="検索結果", interactive=False, wrap=True)
        csv_file = gr.File(label="CSV")

        search_btn.click(
            search_comments,
            [query, search_mode, user, start_date, end_date, kind, limit],
            [results, status, csv_file],
        )
        query.submit(
            search_comments,
            [query, search_mode, user, start_date, end_date, kind, limit],
            [results, status, csv_file],
        )

    with gr.Tab("統計"):
        overview = gr.Markdown()
        overview_btn = gr.Button("DB概要を更新")
        overview_btn.click(summary_stats, outputs=overview)

        n_users = gr.Slider(10, 200, value=50, step=10, label="上位ユーザー数")
        users_btn = gr.Button("ユーザーランキング")
        users_table = gr.Dataframe(interactive=False)
        users_btn.click(top_users, inputs=n_users, outputs=users_table)

        daily_btn = gr.Button("日別推移")
        daily_chart = gr.Plot()
        daily_btn.click(daily_plot, outputs=daily_chart)

    with gr.Tab("SQL"):
        preset = gr.Dropdown(list(SQL_PRESETS), value="最近100件", label="プリセット")
        sql = gr.Code(value=SQL_PRESETS["最近100件"], language="sql", label="SQL")
        preset.change(load_preset, inputs=preset, outputs=sql)
        sql_btn = gr.Button("SQL実行", variant="primary")
        sql_status = gr.Markdown()
        sql_result = gr.Dataframe(interactive=False, wrap=True)
        sql_btn.click(run_readonly_sql, inputs=sql, outputs=[sql_result, sql_status])

demo.queue(default_concurrency_limit=8)
demo.launch(share=True, debug=False)
